## 网络层 · IP 与 MAC：扁平编址 vs 层次编址

> 2026-10-12 · 来源：学习对话（用户自述中的类比，抓得很准）

- **我的理解**（原话）：「这一层就是给网络设备 mac 地址身份证加了一层可以查找的索引，也就是 ip，包括应用层的网站域名也是 ip 地址方便查找的索引，而这一切就是为了每个不同的网络中可以有同 ip 的设备，而且本质上也是多层结构，可以通过和路由表不同位数的匹配来实现更简化和高效的 packet 转发功能，而且相当于将路由功能不断分层下发，减少了核心路由器或者交换机的性能压力」。
- **判定：这段理解基本正确，且抓到了网络层最重要的直觉。** 四个要点逐一对应：
  - **「MAC 是身份证、IP 是可查找的索引」** → MAC **扁平**（无层次、不携带位置信息、烧在网卡、全球唯一）；IP **层次化**（网络号 + 主机号，按**位置**分配，可聚合、可路由）。**层次化是 IP 能聚合的前提。**
  - **「每个不同的网络中可以有同 IP 的设备」**（更准确：私网内可有同 IP）→ 这是 **NAT + 私有地址** 的结果（10/8、172.16/12、192.168/16 可在不同私网重复），背景是 IPv4 地址耗尽。**注意：公网 IP 全网唯一；私网 IP 只在私网内唯一。**
  - **「和路由表不同位数的匹配」** → **最长前缀匹配**（longest prefix match）：路由表存「前缀 / 长度」，转发时选**匹配最长**的那条。这正是 CIDR 的工作方式。
  - **「路由功能不断分层下发，减少核心路由器压力」** → **路由聚合**（route aggregation / CIDR）：一连串连续网络号可合并成一条大前缀，核心路由器只需存大块路由，路由表规模大幅下降。**这就是「分层下发」。**
- **设计层（为什么必须层次化）**：若 IP 像 MAC 一样扁平，全球路由表必须为**每一台主机**存一条，表规模 ~ 主机数（数十亿），路由器内存与查表速度都撑不住。层次化后，**同一网络内的所有主机共享一条路由表项**，且可继续向上聚合。**IP 的可扩展性是「层次化 + 聚合」换来的。**
- **现实对照**：Linux `ip route` 查 FIB（转发信息库）；`ip addr` 看接口地址与掩码。**边界**：408 考概念与计算（子网划分 / CIDR / 最长前缀匹配），不考 FIB 实现。
- **考试侧重点**：① **子网划分 / CIDR 计算**（掩码 ↔ 网络号 / 主机号；可用主机数 2^n - 2；地址块聚合）★核心；② **路由表匹配与转发**（最长前缀匹配，判下一跳）★核心；③ MAC vs IP 对比（层次 / 所属层 / 分配方式 / 是否可路由）。
- **易错点**：把「私网 IP 可重复」说成「公网 IP 可重复」；最长前缀匹配记成「最短」；地址块聚合时算错块大小（必须为 2^n 且对齐）。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13


## 网络层 · 丢包的两大来源，与「缓存」如何把丢包和时延绑在一起

> 2026-10-12 · 来源：学习对话（用户自述判定）

- **我的理解**（原话）：「丢包按刚才的说法一个是链路质量差一个就是路由的缓存不够用吧, 时延也是和路由器的缓存强相关, 但是时延和丢包本质还是有点不同的」。
- **判定**：**全对**，而且抓住了关键（**缓存是耦合点**）。
- **丢包的两大来源**：
  1. **链路误码（信号坏）**：物理层 BER → CRC 不过 → 丢帧（或无线 FEC 后仍残余错误）。
  2. **缓存溢出（拥塞）**：路由器队列满 → **尾丢弃（tail drop）** 或 **AQM**（RED / CoDel）主动丢。
  - 次要来源：路由变化 / 路由黑洞、TTL 耗尽（路由环路）、MTU 分片丢失。
- **时延的四分量**：**发送 + 传播 + 处理 + 排队**。前三个基本固定（传播时延由距离决定），**只有排队时延随负载变化** —— 这就是「时延和路由器缓存强相关」的准确来源。
- **缓存是同一个旋钮，但两个方向相反**：
  - 缓冲区**小** → 丢包多、排队少 → **时延低**。
  - 缓冲区**大** → 丢包少、排队久 → **时延高** —— 即 **Bufferbloat**（大缓存把时延顶到几百 ms ~ s 级）。
- **「时延与丢包本质不同」的准确表述**：
  - **丢包是离散事件**：队列满则丢，非 0 即 1（**跨过容量那一刻**）。
  - **时延是连续量**：队列越长越晚，连续变化。
  - 关系：**时延是「缓冲区占用」的连续度量，丢包是它跨过容量的那一刻** → **时延上升先于丢包出现**（时延是先兆，丢包是后果）。
- **设计层**：这解释了为什么「大缓存」不是纯好事 —— 它把拥塞从「丢包信号」变成「时延黑洞」，反而让**只看丢包的 TCP 更难察觉**（队列要满到丢包，TCP 才降速）。
- **考试侧重点**：**分组丢失的原因**；**时延四分量计算**（★核心）；路由器 / 交换机的队列与丢弃的概念层辨析。
- **易错点**：把「排队时延」归到「传播时延」；以为缓冲区越大越好；把链路误码丢包与拥塞丢包混为一谈（TCP 自己也分不清）。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13

## 网络层 · IP 分片：标识 / DF / MF / 片偏移（补讲）

> 起因：块2 讲 MSS 时只说了「MSS 是为了避免分片」，分片本身一直没讲——那句 MF/DF 是欠账。分片归 **网络层**（IP 首部字段），不在链路层。

### 一、IPv4 首部里与分片有关的三个字段

| 字段 | 位宽 | 作用 |
|---|---|---|
| **标识**（Identification） | 16 | **同一数据报的所有分片共用同一个值**——重组时靠它归并 |
| **标志**（Flags） | 3 | 保留位(0) + **DF** + **MF** |
| **片偏移**（Fragment Offset） | 13 | 本片数据在**原始数据部分**中的起点，**单位 = 8 字节** |

两个标志位的语义（最容易记反）：

- **DF（Don't Fragment）= 1：不许分片。** 若超过 MTU 又不能分片 → **丢弃 + 回 ICMP 差错**（Type 3 / Code 4）。默认通常为 0。
  - **易错**：DF=1 只表示「不许分片」，**不代表这是一个分片**。
- **MF（More Fragments）= 1：后面还有分片。** 所以**只有最后一片 MF = 0**。
  - **易错**：MF 的语义是「**还有后续**」，不是「我是分片」。**未分片的完整数据报 MF = 0**。

### 二、分片的三条规则

1. **除最后一片外，每片的数据长度必须是 8 字节的整数倍** —— 片偏移以 8 字节为单位，否则偏移表达不出来。**最后一片不受此限**。
2. **每片都有自己完整的 IP 首部**（首部被复制）—— 因此分片**放大首部开销**：分 k 片就多花 **(k−1) × 20 B**。
3. **首部里改写的只有三处**：总长度（本片）、MF、片偏移 —— **标识、源 IP、目的 IP、协议等一律原样复制**（重组靠它们归并）。

### 三、算例（408 高频大题的模板）

原始 IP 数据报**总长 4000 B**（首部 20 B → 数据 3980 B），经过 **MTU = 1500** 的链路。

**第 1 步**：每片能装多少数据？`MTU − 首部 = 1500 − 20 = 1480 B`，且 1480 ÷ 8 = 185 是整数 ✓

**第 2 步**：分几片？`⌈3980 / 1480⌉ = 3` 片

**第 3 步**：逐片填表

| 片 | 数据长度 | 总长度 | 片偏移 | MF |
|---|---|---|---|---|
| 1 | 1480 | 1500 | 0 | 1 |
| 2 | 1480 | 1500 | **185** | 1 |
| 3 | 1020 | 1040 | **370** | **0** |

- 偏移 = 该片数据在原始数据中的**起始字节数 ÷ 8**：片 2 起于 1480 → 1480/8 = 185；片 3 起于 2960 → 2960/8 = 370
- 总长 = 20（本片首部）+ 本片数据长度
- 最后一片数据 1020 **不是 8 的倍数也没关系**（它是最后一片）

**校验**：370 × 8 = 2960，2960 + 1020 = 3980 ✓ 三片数据拼起来正好是原始数据段。

### 四、重组：只在目的主机做

- **中间路由器不重组**（IP「尽力而为、无状态」的一部分——路由器不为分片保存任何状态）
- 归并依据：**标识 + 源 IP + 目的 IP + 协议**（四者相同才算同一数据报的片）
- **定时器**：超时仍未凑齐 → **已收到的全部丢弃**
- 转发时**不保证片按序到达**，也不保证走同一条路径

### 五、为什么片偏移的单位偏偏是 8 字节

16 位总长度最大表示 65535 B，而片偏移只有 13 位。若以**字节**为单位，13 位最多表示 8191 B 的位置，**根本盖不住 65535 的报文**。

改成 **8 字节为单位**：13 位最大表示 `8191 × 8 = 65528 B`，**刚好覆盖 65535 的报文**（65528 > 65515 = 65535 − 20）。

**单位是为了「用 13 位装下 16 位的地址空间」**——设计逼出来的，不是随意选的。这也正是规则 1 的来源。

### 六、回扣上一轮：DF 与 PMTUD

块2 说「MSS 是为了避免分片」「PMTUD 探整条路径的最小 MTU」，机制就落在 DF 上：

1. 源主机把 **DF = 1**（不许分片）
2. 路径上某跳 MTU 更小 → 路由器**丢弃 + 回 ICMP「需要分片但 DF 置位」**
3. 源主机据此**调小发送大小**、重试
4. 收敛后得到**路径 MTU** → 反推 MSS

**MSS 协商只解决第一跳，PMTUD 才解决整条路径**；二者配合才真正做到「不给 IP 层分片的机会」。

### 七、408 边界与 IPv6 一句

- **考**：标识 / DF / MF / 片偏移四个字段的位宽与语义；**分片计算题**（分几片、每片总长 / 偏移 / MF）；**片偏移单位 8 字节**；**重组只在目的主机**；分片放大首部开销
- **IPv6 差别**（低频）：**路由器一律不分片**，只有**源主机**能分；分片信息放在**分片扩展首部**，不在基本首部；没有 DF 位；不能分片时回 ICMPv6「Packet Too Big」

### 八、易错点清单

- ❌ 把 MF / DF 记反：**MF=1 是「还有后续」，DF=1 是「不许分片」**
- ❌ 以为**未分片的报文 MF=1** —— 未分片时 **MF = 0**
- ❌ 片偏移按**字节**算 —— 必须 **÷8**
- ❌ 认为每片数据都要是 8 的倍数 —— **只有最后一片除外**
- ❌ 总长度写成原始长度 —— 各片的**总长度是各片自己**的（20 + 本片数据）
- ❌ 以为路由器会重组 —— **重组只在目的主机**
- ❌ 以为分片没有额外开销 —— 分 k 片多花 **(k−1) × 20 B**


## 网络层 · 补讲：分片的第二个陷阱 —— MTU 减首部后不是 8 的倍数

上一节的算例（MTU 1500）里，`1480 ÷ 8 = 185` 恰好整除，**把这个坑藏起来了**。换个 MTU 就露出来了。

### 陷阱

**MTU = 800** 时，每片容量 = `800 − 20 = 780 B`。但 **780 ÷ 8 = 97.5，不是整数**。

**不能按 780 切** —— 否则片偏移 = 780/8 = 97.5 不是整数，**根本填不进 13 位的偏移字段**。

正确做法：**向下取到 8 的倍数**

```
每片数据上限 = ⌊(MTU − 首部) / 8⌋ × 8        （最后一片除外）
            = ⌊780 / 8⌋ × 8 = 97 × 8 = 776 B
```

### 同一道算例换成 MTU = 800

原始数据仍是 3980 B：

- 每片上限 **776 B**
- 片数 = `⌈3980 / 776⌉ = 6`（5 × 776 = 3880，余 100）

| 片 | 数据 | 总长 | 偏移 | MF |
|---|---|---|---|---|
| 1 | 776 | 796 | 0 | 1 |
| 2 | 776 | 796 | **97** | 1 |
| 3 | 776 | 796 | **194** | 1 |
| 4 | 776 | 796 | **291** | 1 |
| 5 | 776 | 796 | **388** | 1 |
| 6 | 100 | 120 | **485** | **0** |

**校验**：485 × 8 = 3880，3880 + 100 = 3980 ✓

偏移的分母是 8：片 2 = 776/8 = 97，片 3 = 1552/8 = 194，…，片 6 = 3880/8 = 485。

### 一句话记住

**先算 `MTU − 首部`，再向下取到 8 的倍数，然后才切。** 上一节那个「整除」是运气，不是常态。


## 网络层 · 块1 IP 地址与子网划分 / CIDR ★核心

前置衔接：本块接「IP vs MAC：扁平编址 vs 层次编址」那条线——**IP 之所以可聚合、可路由，根源就在它的两级结构**。

**必考点清单**

| # | 考点 | 优先级 |
|---|---|---|
| 1 | 子网划分（借位）：子网数 / 每子网可用主机数 | ★核心 |
| 2 | 掩码 ↔ 前缀长度互转（/26 = 255.255.255.192） | ★核心 |
| 3 | 求网络地址 / 广播地址（AND 掩码；广播 = 网络地址 + 块大小 − 1） | ★核心 |
| 4 | CIDR 聚合（连续 **且** 对齐才可合） | ★核心 |
| 5 | 判断两主机是否同网段 | ☆常规 |
| 6 | 地址总数 / 可用主机数（2^(32−n) 与 2^(32−n) − 2） | ☆常规 |

### 1.1 机制层（①）

**（1）IP 地址的二级结构**

32 位 = 4 字节，点分十进制。结构是 **〈网络号 net-id〉〈主机号 host-id〉**：

- **网络号**：分配机构统一分配，标识「在哪个网络」
- **主机号**：本网管理员自行分配，标识「网络里的哪台主机」

**同一网络内的主机共享同一网络号**——这就是「可聚合」的根源。

**（2）分类编址（历史方案，408 仍常考）**

| 类 | 首位模式 | 第 1 字节 | 网络号位数 | 主机号位数 | 可指派网络数 |
|---|---|---|---|---|---|
| A | 0 | 1 – 126 | 8 | 24 | 2⁷ − 2 = **126** |
| B | 10 | 128 – 191 | 16 | 16 | 2¹⁴ − 1 = **16383** |
| C | 110 | 192 – 223 | 24 | 8 | 2²¹ − 1 = **2097151** |
| D | 1110 | 224 – 239 | — | — | 多播（不划分） |
| E | 1111 | 240 – 255 | — | — | 保留 |

减 1（A 类减 2）的原因：**网络号全 0 不指派**；A 类还多一个 **127.x 保留做回环**。

**（3）子网掩码**

32 位，**网络号（含子网号）位全 1、主机号位全 0**。

- `IP ∧ 掩码 = 网络地址`——判断两台主机是否同网段就靠它
- 默认掩码：A /8 = 255.0.0.0，B /16 = 255.255.0.0，C /24 = 255.255.255.0

**子网划分（借位）**：从主机号**高位**借 n 位当子网号

- 子网数 = **2ⁿ**
- 每子网主机位 = 原主机位 − n
- 每子网可用主机数 = **2^(主机位) − 2**
- 结构由二级变**三级**：网络号 + 子网号 + 主机号；**对外仍是同一个网络号**

**（4）CIDR（无分类编址）**

记法 `a.b.c.d/n`，n = 网络前缀长度。

- 取消分类，**前缀长度任意**
- 地址块大小 = **2^(32−n)**
- **地址聚合（构造超网）**：把多个连续小块合成一大块 → 压小路由表（即「最长前缀匹配」的另一半）
- 聚合的两个条件：**地址连续** 且 **起始地址对齐**（能被块大小整除）。少一个都不行。

**（5）掩码速查**

| 前缀 | 掩码 | 块大小 | 可用主机 |
|---|---|---|---|
| /24 | 255.255.255.0 | 256 | 254 |
| /25 | 255.255.255.128 | 128 | 126 |
| /26 | 255.255.255.192 | 64 | 62 |
| /27 | 255.255.255.224 | 32 | 30 |
| /28 | 255.255.255.240 | 16 | 14 |
| /29 | 255.255.255.248 | 8 | 6 |
| /30 | 255.255.255.252 | 4 | 2 |

记法：**第 4 字节里 1 的个数 = n − 24**，2 的幂从 128 往下走。

**（6）特殊地址**

| 地址 | 含义 |
|---|---|
| 0.0.0.0 | 「本网络本主机」（DHCP 请求的源地址）；路由表中表示默认路由 |
| 255.255.255.255 | 受限广播（本网广播，路由器不转发） |
| 127.x.x.x（常用 127.0.0.1） | 回环地址，本机协议栈测试 |
| 主机号全 0 | **网络地址**，代表「这个网络本身」，不能分给主机 |
| 主机号全 1 | **直接广播地址**，如 192.168.1.255/24 |
| 10.0.0.0/8、172.16.0.0/12、192.168.0.0/16 | 私有地址（不路由到公网） |

### 1.2 设计层（②）

**① 为什么地址要分「网络号 + 主机号」两级？—— 为了让路由表能压小。**
路由器只需记「到某网络走哪个口」，不必记每一台主机。若扁平成 32 位随机分配，路由表要存全网主机条目，不可能实现。**这就是「IP 可聚合」的具体机制。**

**② 为什么要有子网划分？—— 大网内部要再分层。**
拿到一个 A/B/C 类大网后，内部常要按部门 / 楼层独立管理，并**隔离广播域**。借主机号做子网号后，**对外仍是一个网络号**（外部路由器只看到外网号），只有本单位内部才细分——这就是三级结构的意义：简化管理、隔离广播、提高安全。

**③ 为什么 CIDR 取代分类编址？—— 分类太浪费。**
B 类给 65534 个地址；实际只要 1000 个地址的单位，拿 B 类浪费 98%，拿 C 类又不够。CIDR 按需分配任意长度前缀、且可聚合，大幅提升利用率并压小路由表。

**④ 为什么「可用主机数 = 2ⁿ − 2」？**
主机号全 0 = 网络地址（表网络本身），全 1 = 广播地址，**两者都不能分配给主机**。

### 1.3 验收层（③）

| # | 题型 | 模板要点 | 优先级 |
|---|---|---|---|
| 1 | 子网划分 | 借 n 位 → 2ⁿ 子网；每子网可用 2^(h−n) − 2 | ★核心 |
| 2 | 掩码 ↔ 前缀互转 | /26 → 255.255.255.192（第 4 字节 1 的个数 = n − 24） | ★核心 |
| 3 | 求网络地址 / 广播地址 | `IP ∧ 掩码`；广播 = 网络地址 + 块大小 − 1 | ★核心 |
| 4 | CIDR 聚合 | 连续 **且** 对齐；块大小翻倍、前缀 −1 | ★核心 |
| 5 | 判断同网段 | 各自 AND 掩码，比网络地址 | ☆常规 |
| 6 | 地址数 / 可用主机数 | 2^(32−n) 与 2^(32−n) − 2 | ☆常规 |

**代表例题**

> 某单位分配到 **200.100.50.0/24**，现需划分为 **4 个等长子网**。
> （1）子网掩码是多少？（2）每个子网可用主机地址多少个？
> （3）写出 4 个子网的网络地址与广播地址。

**解**

（1）4 = 2² → 借 2 位 → 前缀 24 + 2 = **26** → 掩码 **255.255.255.192**（/26）

（2）主机位 = 32 − 26 = 6 → 2⁶ = 64 个地址 → 可用 **64 − 2 = 62** 个

（3）

| 子网 | 网络地址 | 广播地址 | 可用范围 |
|---|---|---|---|
| 1 | 200.100.50.0/26 | 200.100.50.63 | .1 – .62 |
| 2 | 200.100.50.64/26 | 200.100.50.127 | .65 – .126 |
| 3 | 200.100.50.128/26 | 200.100.50.191 | .129 – .190 |
| 4 | 200.100.50.192/26 | 200.100.50.255 | .193 – .254 |

**变式（CIDR 聚合）**

> 路由表有 4 条：192.168.4.0/24、192.168.5.0/24、192.168.6.0/24、192.168.7.0/24，能否聚合成一条？

能 → **192.168.4.0/22**。
检查：4 块**连续**；起始第 3 字节 4 能被 4 整除（4/4 = 1），**正好落在 /22 边界**。覆盖 192.168.4.0 – 192.168.7.255（共 1024 个地址）。

**易错点**

- ❌ 可用主机数忘减 2（网络地址 + 广播地址）
- ❌ 广播地址算成「网络地址 + 块大小」（应为 **+ 块大小 − 1**）
- ❌ 聚合不检查对齐：192.168.1.0/24 + 192.168.2.0/24 **不能**合成 /23（1 是奇数，跨过边界）
- ❌ /26 写成 255.255.255.128（那是 /25）；第 4 字节 1 的个数 = **n − 24**
- ❌ 误以为子网划分后「网络号变了」——**对外仍是同一个网络号**，只在内部细分


## [CS144③] IP 路由器与最长前缀匹配（check6）

> 2026-10-20 · 来源：CS144 项目重修（learning-408/cs144 实读 + check0–7 官方 PDF）

- **我的理解**（Evidence-based）：路由器只回答两个问题 —— 从**哪个接口**发出去、交给**哪个下一跳**。这两个判断全部来自一张 match-action 路由表，不需要任何 TCP / ARP / 以太网的知识。

- **项目映射**（出处：`cs144` 分支 `origin/check7-startercode:src/router.hh`）：`add_route(route_prefix, prefix_length, optional<Address> next_hop, size_t interface_num)` + `route()` + `interface(N)`；官方例子：`18.47.0.0/16` ⇒ prefix = 18×2^24 + 47×2^16 = 305070080，length = 16。默认路由即 `prefix_length == 0`。

- **核心机制**：① 匹配 = 目的地址的最高 N 位与 prefix 的最高 N 位相同；② 在匹配成功的路由里挑 **prefix_length 最大**的（最长前缀匹配）；③ 无匹配 → 丢；④ TTL：本来就是 0 或减完变 0 都丢；⑤ next_hop 为空（直连路由）→ 用数据报的目的 IP 当下一跳；⑥ 递减 TTL 后**必须重算首部校验和**，再交给 `interface(N)->send_datagram()`。

- **为什么这样设计**（check6 PDF 原文大意）：路由器从不关心 TCP、ARP 或以太网帧，甚至不知道链路层长什么样 —— 它只和 Internet datagram 打交道，只通过 `NetworkInterface` 这一层抽象去碰链路层。这就是 check5 被单独拆成一个零件的原因。

- **考试侧重点**：★网络层核心。路由聚合、最长前缀匹配、TTL、IPv4 首部校验和、直接交付 vs 间接交付。

- **易错点**：① **移位 UB** —— `0xffffffff << (32 - prefix_length)`，当 `prefix_length == 0` 时就是把 32 位整数移 32 位，属未定义行为；官方说测试跑 sanitizer 会抓到，**默认路由必须特判**；② 改了 TTL 忘了重算校验和（下一跳直接丢）；③ 直连路由 next_hop 为空的情况；④ `ipv4_numeric()` / `from_ipv4_numeric()` 用反。

- **待解决问题**：无。check6 代码只要 30–60 行，是全系列性价比最高的一个 check。

- **复习计划**：R1 @ 2026-10-21 · R2 @ 2026-10-23 · R3 @ 2026-10-27


## [CS144·验收] 真题对位变式：路由匹配、TTL 与 ARP（§8 题型 5–7）

> 2026-10-22 · 来源：CS144 真题对位变式（自编，题目与解答真源 = `modules/408/CS144.md` §8.2/§8.3；全部答案先手算复核再落盘）

- **用法（对齐 LEARNING.md §六）**：限时作答——选择 2 分钟、大题 15–20 分钟；先遮住【解】自己算，再对答案。卡住的题回 `modules/408/CS144.md` §3.6 / §3.5 回补，并按归因维度记账：概念不清 / 计算失误 / 没记住 / 审题错 / 项目误导 / 存疑待核。
- **归属说明**：教材把 ARP 划在**网络层**（15），而 CS144 check5 的实现视角 cell 已落在 14（[CS144②] 网络接口与 ARP）。两边都算数：本 cell 的题型 7 补的是**教材考点视角**。

**题型 5：最长前缀匹配 + 默认路由 + 直连交付**（★核心 · 一题覆盖 check6 全部语义）

【题】某路由器的路由表：

| 路由 | 下一跳 | 接口 |
|---|---|---|
| 10.0.0.0/8 | （直连） | 0 |
| 10.1.0.0/16 | 10.0.0.5 | 1 |
| 10.1.128.0/17 | 10.0.0.9 | 1 |
| 0.0.0.0/0 | 192.168.1.1 | 2 |

四个目的地址各从哪个接口转发、下一跳是谁？① 10.1.130.5　② 10.1.5.5　③ 10.200.1.1　④ 172.16.0.1

【模板】逐条判 $(\text{dst} \,\&\, \text{mask}) == (\text{prefix} \,\&\, \text{mask})$；在成立的里挑 **prefix_length 最大**的；/0 默认路由永远匹配但优先级最低；**直连路由把目的地址本身当下一跳**。

【解】① 匹配 /8、/16、/17（第三字节 130 ≥ 128，落在 10.1.128.0–10.1.255.255）→ **/17，接口 1，下一跳 10.0.0.9**。
② 匹配 /8、/16（5 < 128，不中 /17）→ **/16，接口 1，下一跳 10.0.0.5**。
③ 匹配 /8（第三字节 200 ≠ 1，不中 /16 与 /17）→ **接口 0，直连，下一跳 = 目的地址 10.200.1.1 本身**。
④ 只有 /0 匹配 → **接口 2，下一跳 192.168.1.1**。

【易错】③ 的「直连时下一跳 = 目的地」与 ④ 的「/0 兜底」各占一半分值；写出 & 掩码比较过程比背结论重要。
【CS144 对位】这正是 `Router::route()` 里 next_hop 为 optional 的语义；掩码构造时 prefix_length == 0 必须特判（32 位移 32 位是 UB，sanitizer 会抓）。

**题型 6：TTL 与首部校验和**（☆常规）

【题】TTL = 1 的分组到达路由器，如何处理？TTL = 2 呢（转发后首部哪个字段变化、还需做什么）？408 视角：前一种情况真实路由器还会做什么（CS144 官方不要求）？

【解】TTL = 1：减 1 后为 0 → **丢弃**；408 还要答：**向源点发 ICMP 时间超时差错报告**（CS144 明说不要求实现，但考点照考——traceroute 正靠它工作）。TTL = 2：正常转发，TTL 变 1，**首部校验和必须重算**（首部任一字段变了都要重算）。

【易错】不是「转发一个 TTL=0 的包」。

**题型 7：ARP 的两种场景**（★核心 · check5 第一号语义坑）

【题】主机 A（192.168.0.10/24，MAC = A）要发数据给同网段 B（192.168.0.20/24）与跨网段 C（10.0.0.1，默认网关 G = 192.168.0.1）。A 的 ARP 缓存为空。
(1) A 要发几个 ARP 请求？请求的「目标 IP 字段」分别是什么？帧的目的 MAC 是什么？谁回应？
(2) A 发往 C 的帧里，目的 IP 与目的 MAC 分别是什么？A 实际解析的是谁的 MAC？
(3) 用 CS144 的接口表述：send_datagram(dgram, next_hop) 的 next_hop 两种场景各取什么？

【模板】**IP 层端到端不变，MAC 层逐跳变；ARP 解析的永远是「下一跳」**（同网段 = 目的主机本身，跨网段 = 默认网关）。

【解】(1) 两个请求：目标 IP 分别为 **192.168.0.20**（B）与 **192.168.0.1**（G）；帧目的 MAC 均为**广播 ff:ff:ff:ff:ff:ff**；只有被询问的那台主机以**单播 reply** 回应（reply 携带自己的 MAC，请求方顺路学习 30 秒）。
(2) 目的 IP = **10.0.0.1（C，端到端不变）**，目的 MAC = **G 的 MAC**；A 解析的是**网关的 MAC**。
(3) 同网段 next_hop = B 的 IP；跨网段 next_hop = G 的 IP——**绝不是 C 的 IP**（check5 官方脚注专门警告的那条混淆）。

- **复习计划**：R1 @ 2026-10-23 · R2 @ 2026-10-25 · R3 @ 2026-10-29